In [1]:
#path setup
#since the notebook is in a subfolder, we need to add the src folder to the path
#The issue can be solved by installing the package in editable mode
from pathlib import Path
import sys
project_root = next(parent
    for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "src" / "ML_LC_Classifier").is_dir())
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

In [2]:
import re
from pathlib import Path
import pandas as pd
from ML_LC_Classifier import assess_map_accuracy
# ---------------------------------------------------------
# 1. Configuration
# Folders containing the rasters
FOLDERS = {
    "sentinel2": Path(r"C:\AFLICM_Test\hybrid_lc\hybrid_outputs\5_final_refined"),
    "landsat": Path(r"C:\AFLICM_Test\hybrid_lc\Landsat_Multi\stage1_output_healed\LandCov\4_final_refined"),
}
# Path to your downloaded GeoJSON testing file
ref_landsat = Path(r"C:\Users\AFahrezi\OneDrive - CIFOR-ICRAF\AFCLIM_TL\stable_GT\TL_Landsat_Samples_with_Stability_final_split.geojson")
ref_sentinel2 = Path(r"C:\Users\AFahrezi\OneDrive - CIFOR-ICRAF\AFCLIM_TL\stable_GT\TL_Sentinel2_Samples_with_Stability_final_split.geojson")
REFERENCE_DATA = {
    "sentinel2": ref_sentinel2,
    "landsat": ref_landsat,
}

# Fields in your GeoJSON
CLASS_ID_FIELD = "Class_id"        # Ground truth class ID column
CLASS_NAME_FIELD = "lc_names"   # Ground truth class name column
# Directory where output reports will be saved
OUTPUT_ROOT = Path("output/batch_accuracy_assessment_final_split_area")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

In [3]:
def get_year_and_filter(filename: str, sensor: str) -> tuple[int | None, str | None]:
    """Detect year from filename and match with available stability column."""
    match = re.search(r"(20\d\d)", filename)
    if not match:
        return None, None
    
    year = int(match.group(1))
    
    if year == 2025:
        # Sentinel-2 has 'stable_2025', while Landsat evaluates all 2025 reference points
        stability_col = "stable_2025" if sensor == "sentinel2" else None
    elif year in (2020, 2015, 2005, 2000):
        stability_col = f"stable_{year}"
    else:
        stability_col = None
        
    return year, stability_col

In [4]:
# ---------------------------------------------------------
# 3. Batch Evaluation Loop
# ---------------------------------------------------------
summary_records = []
for sensor, folder in FOLDERS.items():
    if not folder.exists():
        print(f"Warning: Folder not found: {folder}")
        continue
    # Find all GeoTIFFs in the folder
    raster_files = sorted(
        list(folder.glob("*.tif")) + list(folder.glob("*.tiff"))
    )
    if not raster_files:
        print(f"No GeoTIFF rasters found in: {folder}")
        continue
    print(f"\n=======================================================")
    print(f"Processing {sensor.upper()} rasters ({len(raster_files)} found)")
    print(f"=======================================================")
    for raster_path in raster_files:
        year, filter_col = get_year_and_filter(raster_path.name, sensor)
        prefix = f"{sensor}_{year if year else raster_path.stem}"
        raster_out_dir = OUTPUT_ROOT / sensor / (str(year) if year else raster_path.stem)
        print(f"\nEvaluating: {raster_path.name}")
        print(f"  - Detected Year: {year}")
        print(f"  - Stability Filter: {filter_col} == 1" if filter_col else "  - Stability Filter: None (All Points)")
        try:
            result = assess_map_accuracy(
                raster_path=raster_path,
                reference_data=REFERENCE_DATA[sensor],
                class_field=CLASS_ID_FIELD,
                class_name_field=CLASS_NAME_FIELD,
                filter_col=filter_col,
                filter_val=1,
                compute_area_adjustment=True,  # Olofsson area & area-weighted OA
                output_dir=raster_out_dir,
                output_prefix=prefix,
            )
            # Collect summary row
            metrics = result.overall_metrics
            summary_records.append({
                "sensor": sensor,
                "year": year,
                "raster_name": raster_path.name,
                "samples_evaluated": metrics.get("total_samples"),
                "oa_sample": round(metrics.get("overall_accuracy_sample", 0) * 100, 2),
                "oa_area_weighted": round(metrics.get("overall_accuracy_area_weighted", 0) * 100, 2),
                "balanced_accuracy": round(metrics.get("balanced_accuracy", 0) * 100, 2),
                "f1_macro": round(metrics.get("f1_macro", 0), 4),
                "f1_weighted": round(metrics.get("f1_weighted", 0), 4),
                "g_mean_macro": round(metrics.get("g_mean_macro", 0), 4),
                "kappa": round(metrics.get("kappa", 0), 4),
                "mcc": round(metrics.get("mcc", 0), 4),
            })
            print(f"  -> Done! OA (sample): {metrics.get('overall_accuracy_sample')*100:.2f}% | "
                  f"OA (area-weighted): {metrics.get('overall_accuracy_area_weighted')*100:.2f}% | "
                  f"F1-Macro: {metrics.get('f1_macro'):.4f} | G-Mean: {metrics.get('g_mean_macro'):.4f}")
        except Exception as e:
            print(f"  [ERROR] Failed to evaluate {raster_path.name}: {e}")
# ---------------------------------------------------------
# 4. Save Master Summary Table
# ---------------------------------------------------------
if summary_records:
    summary_df = pd.DataFrame(summary_records).sort_values(by=["sensor", "year"], ascending=[True, False])
    summary_csv = OUTPUT_ROOT / "overall_comparison_summary.csv"
    summary_df.to_csv(summary_csv, index=False)
    
    print("\n=======================================================")
    print("MASTER ACCURACY SUMMARY ACROSS ALL RASTERS:")
    print("=======================================================")
    print(summary_df.to_string(index=False))
    print(f"\nSaved summary table to: {summary_csv}")


Processing SENTINEL2 rasters (2 found)

Evaluating: Final_LULC_S2_2020_clean_all.tif
  - Detected Year: 2020
  - Stability Filter: stable_2020 == 1
  -> Done! OA (sample): 75.50% | OA (area-weighted): 75.58% | F1-Macro: 0.6985 | G-Mean: 0.8210

Evaluating: Final_LULC_S2_2025_clean_all.tif
  - Detected Year: 2025
  - Stability Filter: stable_2025 == 1
  -> Done! OA (sample): 79.91% | OA (area-weighted): 80.51% | F1-Macro: 0.7342 | G-Mean: 0.8474

Processing LANDSAT rasters (4 found)

Evaluating: Landsat_2005_LULC_final_refined.tif
  - Detected Year: 2005
  - Stability Filter: stable_2005 == 1
  -> Done! OA (sample): 74.69% | OA (area-weighted): 74.57% | F1-Macro: 0.6920 | G-Mean: 0.8173

Evaluating: Landsat_2015_LULC_final_refined.tif
  - Detected Year: 2015
  - Stability Filter: stable_2015 == 1
  -> Done! OA (sample): 72.71% | OA (area-weighted): 74.04% | F1-Macro: 0.6674 | G-Mean: 0.7938

Evaluating: Landsat_2020_LULC_final_refined.tif
  - Detected Year: 2020
  - Stability Filter: s